# Ngày 8 · Hồi quy logistic và cross-entropy

- Viết sigmoid ổn định số và log-loss có chặn xác suất
- Tính gradient Xᵀ(σ(Xw) − y)/n và kiểm tra bằng sai phân
- Huấn luyện bằng gradient descent và khớp scikit-learn

**Đọc:**
- ISLR, §4.3 Logistic Regression
- Stanford CS229, §2.1 Logistic regression
- Vũ Hữu Tiệp, Machine Learning cơ bản, Chương 14 Hồi quy logistic
- UC Berkeley CS189, Lecture 10–11 "Logistic Regression"

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. 569 mẫu, 30 đặc trưng.
_bc = load_breast_cancer()
X_bc = _bc.data
y_bc = 1 - _bc.target   # 1 = ác tính (lớp cần phát hiện), 0 = lành tính
feature_names = list(_bc.feature_names)

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

# Một phép chia dùng chung cho cả tuần: 70% train, 30% validation, phân tầng theo nhãn.
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from sklearn.metrics import log_loss as _sk_log_loss
_Xs = X_bc[:, [0, 1]]                      # mean radius, mean texture
_Xs = (_Xs - _Xs.mean(0)) / _Xs.std(0)
_Xb = np.column_stack([np.ones(len(_Xs)), _Xs])
_ref8 = LogisticRegression(penalty=None, max_iter=10000).fit(_Xs, y_bc)
_ref8_loss = _sk_log_loss(y_bc, _ref8.predict_proba(_Xs)[:, 1])
def _no_warning(fn):
    with warnings.catch_warnings():
        warnings.simplefilter('error')
        return fn()
def _num_grad(f, w, eps=1e-6):
    return np.array([(f(w + eps*e) - f(w - eps*e)) / (2*eps) for e in np.eye(len(w))])


## Sigmoid, log-loss, gradient và huấn luyện

Gợi ý: σ(z) = exp(−log(1 + e⁻ᶻ)) dùng np.logaddexp(0, −z) để không tràn số. Chặn p trong [eps, 1−eps] trước khi lấy log.

In [ ]:
def sigmoid(z):
    raise NotImplementedError

def log_loss(y, p, eps=1e-12):
    raise NotImplementedError

def logistic_grad(Xb, y, w):
    # Xb đã có cột 1 ở đầu. Trả gradient của log_loss theo w.
    raise NotImplementedError

def fit_logistic(Xb, y, lr=0.5, steps=3000):
    # Gradient descent từ w = 0.
    raise NotImplementedError


In [ ]:
_report('d8-logistic',
    _check('Sigmoid đúng và không tràn số', lambda: (np.allclose(_no_warning(lambda: sigmoid(np.array([-1000., 0., 1000.]))), [0, .5, 1]), 'Không cảnh báo overflow khi z = ±1000.')),
    _check('Log-loss khớp scikit-learn, hữu hạn khi p = 0 hoặc 1', lambda: (np.isclose(log_loss(np.array([1,0,1,0]), np.array([.9,.2,.6,.4])), _sk_log_loss([1,0,1,0],[.9,.2,.6,.4])) and np.isfinite(log_loss(np.array([1,0]), np.array([0.,1.]))), 'Chặn p trước khi lấy log.')),
    _check('Gradient khớp sai phân trung tâm', lambda: (np.allclose(logistic_grad(_Xb, y_bc, np.array([.2,-.5,.3])), _num_grad(lambda w: log_loss(y_bc, sigmoid(_Xb @ w)), np.array([.2,-.5,.3])), atol=1e-6), 'So tại w = [0,2, −0,5, 0,3].')),
    _check('Huấn luyện đạt loss của scikit-learn (sai lệch ≤ 1e-4)', lambda: (log_loss(y_bc, sigmoid(_Xb @ fit_logistic(_Xb, y_bc))) <= _ref8_loss + 1e-4, f'Loss tham chiếu {_ref8_loss:.5f}.')),
)


## Dự án

Tải Titanic (python py30/download_data.py titanic). Khám phá tỉ lệ sống theo Sex và Pclass, ghi lại các cột có giá trị thiếu. Chưa huấn luyện mô hình; mục tiêu là danh sách cột số, cột phân loại và kế hoạch xử lý thiếu cho ngày 13–14.